### PageIndex — Vectorless RAG

In [ ]:
import os, json, time
from dotenv import load_dotenv
# from langchain.chat_models import init_chat_model   # the generic interface
# from langchain_groq import ChatGroq
from groq import Groq
from pageindex import PageIndexClient

load_dotenv()
llm = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# llm = Groq(
#     api_key=os.getenv("GROQ_API_KEY"),      # or OPENAI_API_KEY, GEMINI_API_KEY, etc.
#     model="openai/gpt-oss-120b"             # any supported model string
# )

pi_client = PageIndexClient(os.getenv("PAGEINDEX_API_KEY"))

# print("PAGE_INDEX_KEY:", os.getenv("PAGEINDEX_API_KEY"))
# print("GROQ_API_KEY:", os.getenv("GROQ_API_KEY"))


In [14]:
PDF_PATH = "./data/international-admission-guide.pdf"   # ← change this
print(f"Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f" Document ID: {doc_id}")


Uploading: ./data/international-admission-guide.pdf
 Document ID: pi-cmuxrroc100370antsoyzijmm


C:\Users\Sanaullah\AppData\Local\Temp\ipykernel_11208\683157120.py:3: UserWarning: Document "international-admission-guide.pdf" was stored as "international-admission-guide_3.pdf".
  result = pi_client.submit_document(PDF_PATH)


In [15]:
# Poll until processing is complete
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.

print("Building tree index...")
print("(This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n Tree index ready!")
        break
    elif status == "failed":
        print("\n Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

Building tree index...
(This runs once per document — the index is cached for reuse)
   Status: completed

 Tree index ready!


In [16]:
# Fetch the full tree
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"Top-level sections: {len(pageindex_tree)}")
print("\nRaw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

Top-level sections: 1

Raw tree (first node):
{
  "title": "2024 Admission Guide",
  "node_id": "0000",
  "start_index": 1,
  "end_index": 6,
  "summary": "The 2024 Admission Guide provides comprehensive information for international undergraduate and postgraduate applicants, detailing course codes, durations, start terms, tuition fees, and international ATAR equivalent scores categorized by study area. It outlines general admission guidelines, CRICOS registration, and policies regarding indicative programs, majors, minors, and specialisations. The document explains academic requirements, including NSW HSC-based assumed knowledge, prerequisites, recommended studies, NESA standards for teaching degrees, and specific mathematics requirements. Additionally, it highlights the Dalyell Scholars stream for high-achieving students, dual degree structures (such as Sciences Po), key definitions and symbols used in the course tables (including additional admission criteria and third-party deliver

In [17]:
# Pretty-print the full tree
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("Full Document Structure:\n")
print_tree(pageindex_tree)

Full Document Structure:

[0000] 2024 Admission Guide  (p.?)
  └─ [0001] 2024 Admission Guide (intro)  (p.?)
  └─ [0002] International students  (p.?)
  └─ [0003] Programs, majors, minors and specialisations  (p.?)
  └─ [0004] Assumed knowledge and prerequisites  (p.?)
  └─ [0005] Recommended studies  (p.?)
  └─ [0006] Dalyell by invitation  (p.?)
  └─ [0007] Key to the tables  (p.?)
  └─ [0008] Prerequisites  (p.?)
  └─ [0009] Explanation of qualification  (p.?)
  └─ [0010] admission scores  (p.?)


In [19]:
# Count total nodes 
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"Total nodes in tree: {total}")
print("Each node = one retrievable section of the document")

Total nodes in tree: 11
Each node = one retrievable section of the document


### LLM Tree Search — The Core of PageIndex

In [45]:
# LLM Tree Search Function

def llm_tree_search(query: str, tree: list, model: str = "openai/gpt-oss-120b") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = llm.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)

In [46]:
#Test with a sample query
query = "Criteria to get admission for international students?"

print(f" Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("Selected Node IDs:", result.get("node_list", []))

 Query: Criteria to get admission for international students?

LLM Reasoning:
The query asks for the admission criteria specifically for international students. The document node titled "International students" (0002) directly mentions a guide to admission criteria for international students, making it the primary source. The node "admission scores" (0010) is described as relating to the same guide and likely contains the detailed criteria (scores, requirements). These two nodes together should cover the needed information.

Selected Node IDs: ['0002', '0010']


### Full End-to-End RAG Pipeline

In [47]:
# Helper: Find nodes by ID

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [57]:
# Generate answer from retrieved nodes

def generate_answer(query: str, nodes: list, model: str = "openai/gpt-oss-120b") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return " No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = llm.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [58]:
# The complete Vectorless RAG function

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f" Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\nAnswer:\n{answer}")
    
    return answer

In [59]:
# Run the full pipeline
answer = vectorless_rag(
    query="What are the syllabus covered in modern llm finetuning?",
    tree=pageindex_tree
)

Query: What are the syllabus covered in modern llm finetuning?

 Reasoning: The query asks about the syllabus for modern LLM finetuning. All nodes in the provided document tree relate to an admission guide, international students, program listings, prerequisites, and related ...
 Retrieved node IDs: []
Sections found: []

Answer:
 No relevant sections found in the document.


In [60]:
# Run the full pipeline
answer = vectorless_rag(
    query="What are the right fit university programs for that offer scholarship also?",
    tree=pageindex_tree
)

Query: What are the right fit university programs for that offer scholarship also?

 Reasoning: The query asks for university programs that are a good fit and also provide scholarships. The node titled 'Programs, majors, minors and specialisations' (node_id 0003) likely lists the available progr...
 Retrieved node IDs: ['0003', '0006']
Sections found: ['Programs, majors, minors and specialisations', 'Dalyell by invitation']

Answer:
The university’s catalogue of **programs, majors, minors and specialisations** (see “Programs, majors, minors and specialisations”) provides the range of study options that can be a right‑fit for a student.  

For scholarship opportunities, the **Dalyell Scholars stream** – described under “Dalyell by invitation” – invites high‑achieving students to receive a scholarship as part of that program.  

So, any of the programs listed in the “Programs, majors, minors and specialisations” section are potential fits, and the Dalyell Scholars invitation offers a sch

In [61]:
# Test with multiple queries
test_queries = [
    "What are the syllabus covered in modern llm finetuning?",
    "What are the syllabus covered in RAG?",
    "Summarize the syllabus of Tokenization Deep Dive?",
]

for q in test_queries:
    print()
    ans = vectorless_rag(q, pageindex_tree, verbose=False)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}...")
    print("-" * 55)


Q: What are the syllabus covered in modern llm finetuning?
A:  No relevant sections found in the document....
-------------------------------------------------------

Q: What are the syllabus covered in RAG?
A:  No relevant sections found in the document....
-------------------------------------------------------

Q: Summarize the syllabus of Tokenization Deep Dive?
A:  No relevant sections found in the document....
-------------------------------------------------------


## Expert-Guided Retrieval